In [7]:
# imports
import torch
import torch.nn as nn
import pandas as pd
from utils import DrivingDataset
from torch.utils.data import DataLoader
from model import DrivingModel
from sklearn.model_selection import train_test_split

In [2]:
# define hyperparameters
num_epochs = 20
batch_size = 8
shuffle = True
lr = 0.0001
early_stopping = 5

In [8]:
# load training data
csv = "raw_data_env_test.csv"
df = pd.read_csv(csv)

# split train, test/val 80-20
train, test_val = train_test_split(
    df,
    random_state = 42,
    test_size = 0.2,
)

# split test, val 50-50
test, val = train_test_split(
    test_val,
    random_state = 42,
    test_size = 0.5,
)

# final train, test, split (80-10-10)
train_ds = DrivingDataset(train) # load training set
test_ds = DrivingDataset(test) # load test set
val_ds = DrivingDataset(val)

# load data
train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle)
test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=shuffle)
val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=shuffle)

FileNotFoundError: [Errno 2] No such file or directory: 'raw_data_env_test.csv'

In [6]:
# check devices from most to least capable
if torch.cuda.is_available():
    dev = 'cuda'
elif torch.mps.is_available():
    dev = 'mps'
else:
    dev = 'cpu'
device = torch.device(dev)

print(f"Running on device: {device}")

# load model to device
model = DrivingModel().to(device)

# MSELoss for continuous values
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Running on device: cuda


In [ ]:
# keep track of metrics through training and validation
train_loss_list = []
train_acc_list = []
val_loss_list = []
val_acc_list = []

# set max value, must be overwritten
best_val_loss = float('inf')

# MARK: start train loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    # uses tqdm for visualization
    for X, y in tqdm(train_ld):

        # load data into device
        X = X.to(device)
        y = y.to(device)

        # zero out gradients, calculate outputs, then loss
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # append training loss to metrics
        running_loss += loss.item()
        total_train += labels.size(0)

    # calculate training metrics per epoch
    epoch_train_loss = running_loss / len(train_ld)
    epoch_train_acc = correct_train / total_train
    train_loss_list.append(epoch_train_loss)
    train_acc_list.append(epoch_train_acc)

    # validation
    model.eval() # set model to evaluation mode
    total_val = 0
    val_running_loss = 0

    # no gradient for validation
    with torch.no_grad():
        for X, y in val_ld:
            # load data into device
            X = X.to(device)
            y = y.to(device)

            # predict
            outputs = model(X)

            # calculate loss
            loss = criterion(X, y)

            # append validation loss to metrics
            val_running_loss += loss.item()
            total_val += y.size(0)

    # calculate validation metrics per epoch
    epoch_val_loss = val_running_loss / len(val_loader)
    epoch_val_acc = correct_val / total_val

    # save for analysis
    val_loss_list.append(epoch_val_loss)
    val_acc_list.append(epoch_val_acc)

    # check if new parameters have better performance
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience = 0
        torch.save(model.state_dict(), 'best_model.pth')
    # if not, terminate early
    else:
        patience += 1
        if patience >= early_stopping:
            print("Early stopping triggered")
            stopped_epoch = epoch
            break